# Ablation: Prompt Strategies

Compares zero-shot, few-shot, and source-aware prompting across languages.

In [ ]:
from sm_sip.config import SigExtConfig
from sm_sip.data import get_test_data
from sm_sip.models import load_sigext_model, load_llm, create_summary_chain, preprocess_dataset
from sm_sip.prompts import get_summary_prompt, SUMMARY_PROMPTS
from sm_sip.pipelines import run_inference, run_evaluation
from sm_sip.utils.io import save_results
from sm_sip.utils.gpu import clear_gpu_memory

In [ ]:
results = {}
for lang in ['it', 'en']:
    sc = SigExtConfig.from_preset(lang, '10k-60t' if lang=='it' else '1k-60t')
    data = get_test_data(lang=lang, num_samples=50, skip_samples=sc.skip_samples)
    sm, st = load_sigext_model(sc.model_id)
    proc = preprocess_dataset(data, sm, st, lang=lang)
    del sm; clear_gpu_memory()
    _, _, pipe = load_llm('meta-llama/Llama-3.1-8B-Instruct', '8bit')
    for pt in SUMMARY_PROMPTS.get(lang, {}).keys():
        chain = create_summary_chain(pipe, get_summary_prompt(lang, pt))
        res = run_inference(proc, chain)
        results[f'{lang}_{pt}'] = run_evaluation(res, lang=lang)
    clear_gpu_memory()
save_results({'ablation': 'prompts', 'results': results}, 'results/ablation_prompts.json')